## GENERAL IMPORTS

In [ ]:
from typing import Final
from pprint import pprint
import pandas as pd

## ``riskquant`` IMPORTS

In [ ]:
from riskquant.config.compile import compile_risk_static_config_from_xlsx
from riskquant.utils.dates import date_str_to_date
from riskquant.market_data.risk_factors.matrix import RiskFactorMatrix

## INPUTS

In [ ]:
# AS OF DATE AS STR
AS_OF_DATE_STR: Final[str] = "2026-01-02"

# Path to the dev data directory
DEV_DATA_DIR: Final[str] = "C:/Users/mharvey/Documents/code/riskquant/data/dev"

# File names of the dev data
DEV_DATA_FILE_NAME_STATIC_CONFIG: Final[str] = "dev_risk_static_data_config_v01.xlsx"
DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS: Final[str] = "dev_risk_factor_levels_v01.xlsx"
DEV_DATA_FILE_NAME_POSITIONS: Final[str] = "dev_positions_v01.xlsx"

In [ ]:
## AS OF DATE NORMALISATION

In [ ]:
as_of_date = date_str_to_date(AS_OF_DATE_STR)

## LOAD RISK STATIC DATA CONFIG: ``RiskStaticConfig``

In [ ]:
# Load and compile the config in one step, since we won't need the intermediate dataframes or frames config after compilation.
cfg = compile_risk_static_config_from_xlsx(
    path=DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_STATIC_CONFIG
    )

In [ ]:
# Basic identity
print('cfg type:', type(cfg))
print('top-level attributes present:')

# Summary counts
print('num curves:', len(cfg.curve_mapping_dict))
print('num risk_factors:', len(cfg.risk_factors_dict))

# Samples
print('\nSample risk factor names (first 20):')
pprint(list(cfg.risk_factors_dict.keys())[:20])

print('\nSample curve names (first 20):')
pprint(list(cfg.curve_mapping_dict.keys())[:20])

# Cached lookups
print('\nSample rf_name_to_id (first 20):')
pprint({k: cfg.rf_name_to_id[k] for k in list(cfg.rf_name_to_id)[:20]})

print('\nSample rf_id_to_risk_factor keys (first 20):')
pprint(list(cfg.rf_id_to_risk_factor.keys())[:20])

# Build absolute mapping for AS_OF_DATE_STR
as_of = pd.to_datetime(AS_OF_DATE_STR).date()
abs_map = cfg.build_risk_factor_name_mapping_for_as_of_date(as_of)
print(f'\nAbsolute->(rel,id) sample for as_of={as_of}:')
pprint(list(abs_map.abs_to_rf_name_and_id.items())[:30])

In [ ]:
# Build RF absolute name to relative name mapping for the as of date.
rf_abs_name_mapping = cfg.build_risk_factor_name_mapping_for_as_of_date(
    as_of_date=as_of_date
    )

# Type. 
print('\nType of rf_abs_name_mapping:', type(rf_abs_name_mapping))

# Display sample of the mapping
print(f'\nSample of absolute to relative name mapping for as_of_date={as_of_date}: \n ')
pprint(list(rf_abs_name_mapping.abs_to_rf_name_and_id.items())[:30])    

## LOAD RISK FACTOR LEVELS INTO AN INSTANCE OF A ``riskquant\market_data\risk_factors\matrix.RiskFactorMatrix``

In [ ]:
# Create RiskFactorMatrix for the as_of_date using the risk factor levels data stored in XLSX.
rfm = RiskFactorMatrix.from_xlsx(
    path=DEV_DATA_DIR + "/" + DEV_DATA_FILE_NAME_RISK_FACTOR_LEVELS, 
    as_of_date=as_of_date, 
    name_to_id_map=rf_abs_name_mapping.abs_to_rf_id, 
    )


# Display basic info about the RFM.
print('\nType of RiskFactorMatrix:', type(rfm))
print('RFM shape:', rfm.shape) 

print('\nRMF Head (first 5 rows):')
print(rfm.head(5).T.to_string())
print('\nRMF Tail (last 5 rows):')  
print(rfm.tail(5).T.to_string())

print("\nPlot of RFM values:")
rfm.plot()